# 0 · Suppress Warnings

In [3]:
import os, warnings, logging

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
    absl.logging.use_absl_handler()
except ImportError:
    pass

logging.getLogger("tensorflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ Warnings suppressed.")


✅ Warnings suppressed.


# 1 · Setup and Paths

In [4]:
import gc, random, pathlib

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.exceptions import ConvergenceWarning

print("TF version:", tf.__version__)
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE        = "/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset"
images_root = os.path.join(BASE, "images")

if "Images" in os.listdir(images_root):
    DATA_DIR = os.path.join(images_root, "Images")
else:
    DATA_DIR = images_root

print("✅ DATA_DIR:", DATA_DIR)
print("Example entries:", os.listdir(DATA_DIR)[:6])

IMG_SIZE   = 331
BATCH_SIZE = 32
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.10


TF version: 2.19.0
✅ DATA_DIR: /kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images
Example entries: ['n02091635-otterhound', 'n02102318-cocker_spaniel', 'n02101388-Brittany_spaniel', 'n02088094-Afghan_hound', 'n02085936-Maltese_dog', 'n02104365-schipperke']


# 2 · Build Datasets

75 / 15 / 10 train/val/test split from breed subdirectories.

**Important:** `train_ds` is shuffled (for future neural net use).
`extract_*_ds` are **unshuffled** — their order matches `y_train/val/test` exactly,
which is required for correct feature-label alignment.

In [6]:
all_classes = sorted(os.listdir(DATA_DIR))
num_classes = len(all_classes)
class_to_idx = {c: i for i, c in enumerate(all_classes)}
print("num_classes:", num_classes)

all_paths, all_labels = [], []
for cls in all_classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):          # sorted for determinism
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(class_to_idx[cls])

total = len(all_paths)
print(f"Total images: {total}")

# Deterministic shuffle
rng = np.random.default_rng(SEED)
idx = rng.permutation(total)
all_paths  = [all_paths[i]  for i in idx]
all_labels = [all_labels[i] for i in idx]

n_test  = int(total * TEST_SPLIT)
n_val   = int(total * VAL_SPLIT)

test_paths,  test_labels  = all_paths[:n_test],             all_labels[:n_test]
val_paths,   val_labels   = all_paths[n_test:n_test+n_val], all_labels[n_test:n_test+n_val]
train_paths, train_labels = all_paths[n_test+n_val:],       all_labels[n_test+n_val:]

print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

# Labels as numpy arrays — order matches extract_*_ds (unshuffled)
y_train = np.array(train_labels)
y_val   = np.array(val_labels)
y_test  = np.array(test_labels)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

def make_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Shuffled — for neural net training (not used for feature extraction)
train_ds = make_ds(train_paths, train_labels, shuffle=True)
val_ds   = make_ds(val_paths,   val_labels,   shuffle=False)
test_ds  = make_ds(test_paths,  test_labels,  shuffle=False)

# Unshuffled — for feature extraction; order matches y_train / y_val / y_test
extract_train_ds = make_ds(train_paths, train_labels, shuffle=False)
extract_val_ds   = make_ds(val_paths,   val_labels,   shuffle=False)
extract_test_ds  = make_ds(test_paths,  test_labels,  shuffle=False)

print("✅ Datasets ready.")


num_classes: 120
Total images: 20580
Train: 15435  Val: 3087  Test: 2058
✅ Datasets ready.


# 3 · Define Backbones

Four ImageNet-pretrained CNNs used as frozen feature extractors.
Preprocessing is applied **inside the dataset pipeline** (not inside the model)
to avoid Keras 3 symbolic tensor issues.

In [7]:
from tensorflow.keras.applications import (
    InceptionV3, InceptionResNetV2, NASNetLarge, EfficientNetB3,
    inception_v3, inception_resnet_v2, nasnet, efficientnet,
)

BACKBONES = [
    ("InceptionV3",       InceptionV3,       inception_v3.preprocess_input),
    ("InceptionResNetV2", InceptionResNetV2, inception_resnet_v2.preprocess_input),
    ("NASNetLarge",       NASNetLarge,       nasnet.preprocess_input),
    ("EfficientNetB3",    EfficientNetB3,    efficientnet.preprocess_input),
]

def make_feature_extractor(model_cls, img_size=IMG_SIZE):
    """Return a plain frozen base model. Preprocessing is handled in the dataset."""
    base = model_cls(
        include_top=False,
        weights="imagenet",
        input_shape=(img_size, img_size, 3),
        pooling="avg",
    )
    base.trainable = False
    return base

def make_preproc_ds(base_ds, preprocess_fn):
    """Apply backbone preprocessing and strip labels so predict() receives only images."""
    return base_ds.map(
        lambda x, y: preprocess_fn(tf.cast(x, tf.float32)),
        num_parallel_calls=AUTOTUNE,
    )

print("Backbone definitions ready.")


Backbone definitions ready.


# 4 · Extract and Save Features

Each backbone's features are saved to `.npy` immediately after extraction.
The model is deleted and memory is freed before loading the next backbone.

Uses `extract_*_ds` (unshuffled) so feature order matches `y_train/val/test`.

In [9]:
FEAT_DIR = "/kaggle/working/features"
os.makedirs(FEAT_DIR, exist_ok=True)

for name, model_cls, preprocess_fn in BACKBONES:
    train_path = os.path.join(FEAT_DIR, f"{name}_train.npy")
    val_path   = os.path.join(FEAT_DIR, f"{name}_val.npy")
    test_path  = os.path.join(FEAT_DIR, f"{name}_test.npy")

    if os.path.exists(train_path):
        print(f"⏭️  {name} already extracted, skipping.")
        continue

    print(f"\n=== Extracting: {name} ===")
    feat_model = make_feature_extractor(model_cls, IMG_SIZE)

    # Unshuffled extraction datasets — order matches y_train / y_val / y_test
    t_ds = make_preproc_ds(extract_train_ds, preprocess_fn)
    v_ds = make_preproc_ds(extract_val_ds,   preprocess_fn)
    e_ds = make_preproc_ds(extract_test_ds,  preprocess_fn)

    train_feats = feat_model.predict(t_ds, verbose=1)
    val_feats   = feat_model.predict(v_ds, verbose=1)
    test_feats  = feat_model.predict(e_ds, verbose=1)

    # Sanity check before saving
    assert train_feats.ndim == 2,                  "❌ Wrong shape — predict returned a tuple"
    assert not np.isnan(train_feats).any(),         "❌ NaNs detected in features"
    assert train_feats.shape[0] == len(y_train),   "❌ Feature count != label count"
    print(f"  shape={train_feats.shape}  mean={train_feats.mean():.4f}  std={train_feats.std():.4f}")

    np.save(train_path, train_feats)
    np.save(val_path,   val_feats)
    np.save(test_path,  test_feats)

    del feat_model, t_ds, v_ds, e_ds, train_feats, val_feats, test_feats
    gc.collect()
    print(f"✅ {name} saved.")

print("\n✅ All backbones extracted.")



=== Extracting: InceptionV3 ===
483/483 ━━━━━━━━━━━━━━━━━━━━ 88s 170ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 19s 197ms/step
65/65 ━━━━━━━━━━━━━━━━━━━━ 14s 222ms/step
  shape=(15435, 2048)  mean=0.1956  std=0.2306
✅ InceptionV3 saved.

=== Extracting: InceptionResNetV2 ===
219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 111s 201ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 24s 248ms/step
65/65 ━━━━━━━━━━━━━━━━━━━━ 18s 279ms/step
  shape=(15435, 1536)  mean=0.1475  std=0.1946
✅ InceptionResNetV2 saved.

=== Extracting: NASNetLarge ===
343610240/343610240 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 281s 497ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 72s 742ms/step
65/65 ━━━━━━━━━━━━━━━━━━━━ 57s 886ms/step
  shape=(15435, 4032)  mean=0.0716  std=0.1797
✅ NASNetLarge saved.

=== Extracting: EfficientNetB3 ===
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 70s 115ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 18s 190ms/step
65/65 ━━━━━━━━━━━━━━

# 5 · Sanity Check + Concatenate Features

In [10]:
print("=== Feature sanity check ===")
for name, *_ in BACKBONES:
    f = np.load(os.path.join(FEAT_DIR, f"{name}_train.npy"))
    print(f"{name}: shape={f.shape}  mean={f.mean():.4f}  std={f.std():.4f}  "
          f"min={f.min():.4f}  max={f.max():.4f}  NaNs={np.isnan(f).any()}")

train_parts = [np.load(os.path.join(FEAT_DIR, f"{name}_train.npy")) for name, *_ in BACKBONES]
val_parts   = [np.load(os.path.join(FEAT_DIR, f"{name}_val.npy"))   for name, *_ in BACKBONES]
test_parts  = [np.load(os.path.join(FEAT_DIR, f"{name}_test.npy"))  for name, *_ in BACKBONES]

X_train = np.concatenate(train_parts, axis=1)
X_val   = np.concatenate(val_parts,   axis=1)
X_test  = np.concatenate(test_parts,  axis=1)

del train_parts, val_parts, test_parts
gc.collect()

print(f"\nX_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")


=== Feature sanity check ===
InceptionV3: shape=(15435, 2048)  mean=0.1956  std=0.2306  min=0.0000  max=6.3691  NaNs=False
InceptionResNetV2: shape=(15435, 1536)  mean=0.1475  std=0.1946  min=0.0000  max=4.6752  NaNs=False
NASNetLarge: shape=(15435, 4032)  mean=0.0716  std=0.1797  min=0.0000  max=10.6038  NaNs=False
EfficientNetB3: shape=(15435, 1536)  mean=-0.0005  std=0.2321  min=-0.2610  max=3.8105  NaNs=False

X_train: (15435, 9152)
X_val:   (3087, 9152)
X_test:  (2058, 9152)


# 6 · Scale + PCA

`StandardScaler` is applied before PCA because the four backbones
produce features at different scales. Skipping this step destroys PCA quality.
Both scaler and PCA are fit on training data only.

In [11]:
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

del X_train, X_val, X_test
gc.collect()

N_COMPONENTS = 256

print(f"Applying PCA (n_components={N_COMPONENTS})...")
pca = PCA(n_components=N_COMPONENTS, whiten=True, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

explained = pca.explained_variance_ratio_.sum()
print(f"✅ PCA done — variance explained: {explained:.2%}")
print(f"X_train_pca: {X_train_pca.shape}")


Scaling features...
Applying PCA (n_components=256)...
✅ PCA done — variance explained: 75.84%
X_train_pca: (15435, 256)


# 7 · Train Classifier

`dual=False` is required when n_samples > n_features — converges much faster.
Falls back to LogisticRegression automatically if LinearSVC fails to converge.

In [12]:
# Re-enable convergence warnings just for this cell to detect non-convergence
warnings.filterwarnings("error", category=ConvergenceWarning)

print("Fitting classifier...")
try:
    clf = LinearSVC(C=0.1, max_iter=10000, dual=False, random_state=SEED)
    clf.fit(X_train_pca, y_train)
    print("✅ LinearSVC converged.")
except ConvergenceWarning:
    print("⚠️  LinearSVC did not converge — falling back to LogisticRegression")
    clf = LogisticRegression(
        C=1.0, max_iter=1000, solver="lbfgs",
        multi_class="multinomial", n_jobs=-1, random_state=SEED,
    )
    clf.fit(X_train_pca, y_train)
    print("✅ LogisticRegression fitted.")

warnings.filterwarnings("ignore", category=ConvergenceWarning)

svm_clf = clf

train_acc = accuracy_score(y_train, svm_clf.predict(X_train_pca))
val_acc   = accuracy_score(y_val,   svm_clf.predict(X_val_pca))

print(f"\nTrain accuracy : {train_acc:.4f}")
print(f"Val   accuracy : {val_acc:.4f}")


Fitting classifier...
✅ LinearSVC converged.

Train accuracy : 0.9852
Val   accuracy : 0.9423


# 8 · Evaluate on Test Set

Held out entirely until this point.

In [13]:
y_test_pred = svm_clf.predict(X_test_pca)
test_acc    = accuracy_score(y_test, y_test_pred)
print(f"Test accuracy: {test_acc:.4f}")

report = classification_report(
    y_test, y_test_pred,
    target_names=all_classes,
    output_dict=True,
    zero_division=0,
)
df_report = pd.DataFrame(report).T

print("\n── Worst 10 classes by F1 ──")
print(df_report.iloc[:-3].sort_values("f1-score").head(10)
      [["precision", "recall", "f1-score", "support"]])

print("\n── Best 10 classes by F1 ──")
print(df_report.iloc[:-3].sort_values("f1-score", ascending=False).head(10)
      [["precision", "recall", "f1-score", "support"]])


Test accuracy: 0.9480

── Worst 10 classes by F1 ──
                                          precision    recall  f1-score  \
n02109961-Eskimo_dog                       0.538462  0.636364  0.583333   
n02110185-Siberian_husky                   0.750000  0.625000  0.681818   
n02098413-Lhasa                            0.846154  0.687500  0.758621   
n02106030-collie                           0.909091  0.666667  0.769231   
n02094433-Yorkshire_terrier                0.750000  0.857143  0.800000   
n02086240-Shih-Tzu                         0.736842  0.875000  0.800000   
n02110063-malamute                         0.812500  0.812500  0.812500   
n02089867-Walker_hound                     0.785714  0.846154  0.814815   
n02093428-American_Staffordshire_terrier   0.789474  0.882353  0.833333   
n02106166-Border_collie                    0.733333  1.000000  0.846154   

                                          support  
n02109961-Eskimo_dog                         11.0  
n02110185-Siberian

# 9 · Single Image Inference

All four feature extractors are built once and cached. Preprocessing is applied per backbone before extraction.

In [14]:
# Build and cache all feature extractors once
_feat_models = {
    name: make_feature_extractor(model_cls, IMG_SIZE)
    for name, model_cls, _ in BACKBONES
}
_preproc_fns = {name: preprocess_fn for name, _, preprocess_fn in BACKBONES}

def predict_image(img_path, top_k=5):
    """Return top-k (breed, confidence) pairs for a single image path."""
    img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    x   = tf.keras.utils.img_to_array(img)[np.newaxis, ...]   # (1, H, W, 3)

    feat_vecs = []
    for name, *_ in BACKBONES:
        x_proc = _preproc_fns[name](tf.cast(x, tf.float32))
        feats  = _feat_models[name].predict(x_proc, verbose=0)
        feat_vecs.append(feats)

    x_full  = np.concatenate(feat_vecs, axis=1)
    x_scaled = scaler.transform(x_full)
    x_pca   = pca.transform(x_scaled)

    # Use decision_function scores (LinearSVC) or predict_proba (LogisticRegression)
    if hasattr(svm_clf, "predict_proba"):
        scores = svm_clf.predict_proba(x_pca)[0]
    else:
        scores = svm_clf.decision_function(x_pca)[0]

    top = np.argsort(scores)[::-1][:top_k]
    return [(all_classes[i], float(scores[i])) for i in top]

# ── Test it ───────────────────────────────────────────────────────────────
sample_cls = all_classes[0]
sample_img = os.path.join(DATA_DIR, sample_cls,
                           sorted(os.listdir(os.path.join(DATA_DIR, sample_cls)))[0])
print("Image      :", sample_img)
print("True class :", sample_cls)
print("\nTop-5 predictions:")
for breed, score in predict_image(sample_img):
    print(f"  {breed:<45} {score:.4f}")


Image      : /kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images/n02085620-Chihuahua/n02085620_10074.jpg
True class : n02085620-Chihuahua

Top-5 predictions:
  n02085620-Chihuahua                           1.8511
  n02091467-Norwegian_elkhound                  -0.9992
  n02092339-Weimaraner                          -1.0023
  n02108089-boxer                               -1.0281
  n02111129-Leonberg                            -1.0315
